# Introduction to Social Simulation

## From factors to actors with Rock–Paper–Scissors

Social science often begins with a table: rows are cases, columns are variables, and an outcome is related to explanatory factors. This is useful—but it leaves a question unanswered:

> **What process generated the observed association?**

This notebook moves through three representations of the same tournament:

1. a CSV table and the **factor paradigm**;
2. dictionaries that recover **actors and interactions**;
3. objects that complete **Axtell's agent architecture**.

# Part I — The factor paradigm

We begin as if the tournament had already occurred. The dataset contains one row per player:

- **X:** `preferred_move`;
- **X:** `decision_rule`;
- **Y:** `final_points`.

The decision rules are:

- `always_preferred`: always play the preferred move;
- `never_paper`: randomly choose Rock or Scissors;
- `mostly_preferred`: favor the preferred move but occasionally choose another.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

players_data = pd.read_csv('rps_players.csv')
players_data

## Relating factors to an outcome

We can ask whether players with different attributes obtained different final scores:

$$	ext{final points}_i=f(	ext{preferred move}_i,	ext{decision rule}_i).$$

Group averages provide a simple factor-based analysis.

In [ ]:
mean_by_move = (
    players_data.groupby('preferred_move', as_index=False)
    ['final_points'].mean()
    .sort_values('final_points', ascending=False)
)

mean_by_move

In [ ]:
mean_by_rule = (
    players_data.groupby('decision_rule', as_index=False)
    ['final_points'].mean()
    .sort_values('final_points', ascending=False)
)

mean_by_rule

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(mean_by_move['preferred_move'], mean_by_move['final_points'])
axes[0].set_title('Average points by preferred move')
axes[0].set_ylabel('Average final points')

axes[1].bar(mean_by_rule['decision_rule'], mean_by_rule['final_points'])
axes[1].set_title('Average points by decision rule')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## What this approach gives us

The table allows us to:

- summarize many observations compactly;
- compare outcomes across player attributes;
- identify systematic associations;
- use familiar statistical models and visualizations.

In this tournament, preferred move and decision rule are associated with final points.

But association is not yet a causal explanation. The table does not show **how the points were produced**.

## What the player-level table lacks

A player's points were not produced by that player alone. Each point depended on an encounter with another player.

The CSV does not preserve:

- whom each player encountered;
- what both players selected in each game;
- the order of interactions;
- the random choices generated by behavioral rules;
- the mechanism connecting two decisions to points.

Therefore, a more accurate representation is:

$$Y_i=f(	ext{actor}_i,	ext{opponents},	ext{interactions},	ext{environment}).$$

The factors approach describes an association. To explain its production, we must recover the actors and interactions behind the rows.

# Part II — From factors to actors with dictionaries

We now reconstruct the process that generated the CSV. A Python dictionary makes each actor's state explicit.

At this stage:

- dictionaries contain **agent states**;
- functions specify **behavior**;
- the payoff table specifies the **environment**;
- the tournament schedule specifies **interaction**.

In [ ]:
from random import Random
from itertools import combinations

strategies = ['Rock', 'Paper', 'Scissors']

payoff = {
    ('Rock', 'Paper'): (0, 1),
    ('Paper', 'Rock'): (1, 0),
    ('Rock', 'Scissors'): (1, 0),
    ('Scissors', 'Rock'): (0, 1),
    ('Paper', 'Scissors'): (0, 1),
    ('Scissors', 'Paper'): (1, 0),
    ('Rock', 'Rock'): (0, 0),
    ('Paper', 'Paper'): (0, 0),
    ('Scissors', 'Scissors'): (0, 0)
}

In [ ]:
society = [
    {
        'name': row.player,
        'preferred_move': row.preferred_move,
        'decision_rule': row.decision_rule,
        'current_move': None,
        'score': 0
    }
    for row in players_data.itertuples(index=False)
]

society[:3]

## Behavioral rules

A decision rule converts an agent's stored state into an action. Notice that the rule is now outside the dictionary: the dictionary stores data but does not itself behave.

In [ ]:
def choose_move(agent, rng):
    rule = agent['decision_rule']
    preferred = agent['preferred_move']

    if rule == 'always_preferred':
        move = preferred
    elif rule == 'never_paper':
        move = rng.choice(['Rock', 'Scissors'])
    elif rule == 'mostly_preferred':
        move = rng.choice([
            preferred, preferred, preferred,
            'Rock', 'Paper', 'Scissors'
        ])
    else:
        raise ValueError(f'Unknown decision rule: {rule}')

    agent['current_move'] = move
    return move

## Interaction

One game joins two actors, their decisions, the environmental payoff rule, and state updating.

In [ ]:
def play_game(player1, player2, payoff, rng):
    move1 = choose_move(player1, rng)
    move2 = choose_move(player2, rng)
    points1, points2 = payoff[move1, move2]

    player1['score'] += points1
    player2['score'] += points2

    return {
        'player1': player1['name'],
        'move1': move1,
        'player2': player2['name'],
        'move2': move2,
        'points1': points1,
        'points2': points2
    }

## Reconstructing the tournament

During each of 20 rounds, every player encounters every other player once. The model stores every game before aggregating the resulting scores.

In [ ]:
rng = Random(123)
interaction_history = []

for round_number in range(1, 21):
    for player1, player2 in combinations(society, 2):
        event = play_game(player1, player2, payoff, rng)
        event['round'] = round_number
        interaction_history.append(event)

interaction_history = pd.DataFrame(interaction_history)
interaction_history.head()

In [ ]:
simulated_results = pd.DataFrame(society)[
    ['name', 'preferred_move', 'decision_rule', 'score']
]

simulated_results

The final scores reproduce the CSV because the CSV was generated by this process with the same rules, schedule, and random seed.

The difference is epistemological:

- the CSV contains the **association** between X and Y;
- the simulation contains a **candidate mechanism** capable of generating Y.

We can now inspect the interactions hidden by the player-level table.

In [ ]:
interaction_history.query(
    "player1 == 'Ava' or player2 == 'Ava'"
).head(10)

Ava and Julia have the same preferred move and the same decision rule, yet they receive different final scores. The factor values do not determine the outcome by themselves: their opponents' stochastic choices also matter.

In [ ]:
simulated_results.query("name in ['Ava', 'Julia']")

# Part III — OOP and Axtell's architecture

Dictionaries made agent states visible, but states and behavioral functions remained separate. Axtell describes agents as software objects that combine:

$$	ext{agent object}=	ext{states}+	ext{behavioral rules}.$$

States and rules may be public or private. In Python, a leading underscore indicates private implementation by convention.

In [ ]:
class Player:
    def __init__(self, name, preferred_move, decision_rule):
        # Public states
        self.name = name
        self.preferred_move = preferred_move
        self.current_move = None
        self.score = 0

        # Private state
        self._decision_rule = decision_rule

    # Private behavior
    def _select_from_rule(self, rng):
        if self._decision_rule == 'always_preferred':
            return self.preferred_move
        if self._decision_rule == 'never_paper':
            return rng.choice(['Rock', 'Scissors'])
        if self._decision_rule == 'mostly_preferred':
            return rng.choice([
                self.preferred_move,
                self.preferred_move,
                self.preferred_move,
                'Rock', 'Paper', 'Scissors'
            ])
        raise ValueError(f'Unknown decision rule: {self._decision_rule}')

    # Public behavior
    def choose_move(self, rng):
        self.current_move = self._select_from_rule(rng)
        return self.current_move

    # Public behavior
    def receive_points(self, points):
        self.score += points

A class is written once but can create many agents. Every instance has its own state while sharing the same behavioral repertoire.

In [ ]:
Ava = Player('Ava', 'Rock', 'always_preferred')
vars(Ava)

## Interaction using agent objects

The payoff remains outside the agents because it belongs to the game environment. The interaction procedure connects the two agents to that environment.

In [ ]:
def play_object_game(player1, player2, payoff, rng):
    move1 = player1.choose_move(rng)
    move2 = player2.choose_move(rng)
    points1, points2 = payoff[move1, move2]

    player1.receive_points(points1)
    player2.receive_points(points2)

    return {
        'player1': player1.name,
        'move1': move1,
        'player2': player2.name,
        'move2': move2,
        'points1': points1,
        'points2': points2
    }

## The population object

Axtell also represents the population as an object. It stores agents, manages interaction, keeps the model clock, and computes statistics.

In [ ]:
class PlayerPopulation:
    def __init__(self, data, seed=123):
        # Private population states
        self._players = [
            Player(
                row.player,
                row.preferred_move,
                row.decision_rule
            )
            for row in data.itertuples(index=False)
        ]
        self._rng = Random(seed)
        self._round = 0

    # Public population state
    @property
    def number_of_players(self):
        return len(self._players)

    # Public population behavior
    def agents_interact(self, payoff):
        self._round += 1
        events = []

        for player1, player2 in combinations(self._players, 2):
            event = play_object_game(
                player1, player2, payoff, self._rng
            )
            event['round'] = self._round
            events.append(event)

        return events

    # Public population behavior
    def compute_statistics(self):
        scores = pd.Series([player.score for player in self._players])
        return {
            'round': self._round,
            'mean_score': scores.mean(),
            'minimum_score': scores.min(),
            'maximum_score': scores.max()
        }

    def agent_table(self):
        return pd.DataFrame([
            {
                'player': player.name,
                'preferred_move': player.preferred_move,
                'decision_rule': player._decision_rule,
                'final_points': player.score
            }
            for player in self._players
        ])

## Axtell's typical agent-oriented program

The completed model follows the architecture:

$$	ext{initialize agents}ightarrow	ext{agents interact}ightarrow	ext{compute statistics}ightarrow	ext{repeat}.$$

In [ ]:
population = PlayerPopulation(players_data, seed=123)

object_events = []
statistics = []

for round_number in range(20):
    object_events.extend(population.agents_interact(payoff))
    statistics.append(population.compute_statistics())

object_events = pd.DataFrame(object_events)
statistics = pd.DataFrame(statistics)

In [ ]:
population.agent_table()

In [ ]:
statistics.tail()

## The completed architecture

| Axtell's component | RPS implementation |
|---|---|
| Agent public states | `name`, `preferred_move`, `current_move`, `score` |
| Agent private state | `_decision_rule` |
| Agent private behavior | `_select_from_rule()` |
| Agent public behavior | `choose_move()`, `receive_points()` |
| Environment | `payoff` |
| Interaction | `play_object_game()` |
| Population state | `_players`, `_round` |
| Population behavior | `agents_interact()`, `compute_statistics()` |
| Model output | interaction history, agent table, statistics |

# What changed epistemologically?

The same final table can now be viewed in two ways:

- **Factors:** Which player attributes are associated with higher scores?
- **Actors:** What decisions and interactions generated those scores?

This is the connection to Macy and Willer's movement from factors to actors. Axtell supplies the computational architecture for representing those actors.

The ABM does **not** automatically prove real-world causation. It establishes a generative claim inside the model:

> Given these agents, behavioral rules, interactions, and environmental rules, the observed outcome can be generated.

Empirical evidence is still needed to determine whether that mechanism credibly represents the social world.